In [ ]:
#!/usr/bin/env python3
"""
filter_tracks_by_roi.py
 
Filter tracking CSV files by ROI mask in the first frame.
 
This script prompts you for:
  - The folder path containing your ROI TIFF and CSV files
  - (Optionally) The ROI mask filename
  - (Optionally) The suffix to append to filtered track filenames
 
It matches each spots CSV to its corresponding tracks CSV by leading numeric prefix,
identifies which tracks originate within the ROI in the first frame, and
saves filtered track CSVs with your chosen suffix.
"""

import os
import re
import glob
import pandas as pd
import tifffile
 
def filter_tracks_by_roi(folder_path, roi_file=None, suffix="_filtered_tracks"):
    # Locate ROI mask TIFF
    if roi_file:
        mask_path = os.path.join(folder_path, roi_file)
        if not os.path.isfile(mask_path):
            raise FileNotFoundError(f"Specified ROI file not found: {mask_path}")
    else:
        tiff_files = glob.glob(os.path.join(folder_path, "*.tif")) + glob.glob(os.path.join(folder_path, "*.tiff"))
        if not tiff_files:
            raise FileNotFoundError("No TIFF file found in folder for ROI mask. Provide one.")
        if len(tiff_files) > 1:
            print(f"Multiple TIFFs found, using first: {os.path.basename(tiff_files[0])}")
        mask_path = tiff_files[0]
 
    # Load ROI mask as binary array
    mask = tifffile.imread(mask_path).astype(bool)
    print(f"Loaded ROI mask: {os.path.basename(mask_path)} (shape {mask.shape})")
 
    # Find spot and track CSVs
    spots_files = glob.glob(os.path.join(folder_path, "*spots*.csv"))
    tracks_files = glob.glob(os.path.join(folder_path, "*tracks*.csv"))
 
    # Helper to extract leading numeric prefix
    def get_prefix(path):
        name = os.path.basename(path)
        m = re.match(r"^(\d+)", name)
        return m.group(1) if m else None
 
    spots_map = {get_prefix(f): f for f in spots_files if get_prefix(f)}
    tracks_map = {get_prefix(f): f for f in tracks_files if get_prefix(f)}
 
    # Process each matching pair
    for prefix, tfile in tracks_map.items():
        sfile = spots_map.get(prefix)
        if not sfile:
            print(f"Skipping {os.path.basename(tfile)}: no matching spots CSV.")
            continue
 
        print(f"\nProcessing files for prefix {prefix}:")
        print(f"  Spots:  {os.path.basename(sfile)}")
        print(f"  Tracks: {os.path.basename(tfile)}")
 
        # Read with low_memory to avoid dtype mixing; then explicitly convert types
        spots = pd.read_csv(sfile, low_memory=False)
        tracks = pd.read_csv(tfile, low_memory=False)
 
        # Convert key columns to numeric, drop invalid rows
        for col in ['FRAME', 'POSITION_X', 'POSITION_Y', 'TRACK_ID']:
            if col not in spots.columns:
                raise KeyError(f"Column '{col}' not found in {sfile}")
            spots[col] = pd.to_numeric(spots[col], errors='coerce')
        spots = spots.dropna(subset=['FRAME', 'POSITION_X', 'POSITION_Y', 'TRACK_ID'])
        spots['FRAME'] = spots['FRAME'].astype(int)
        spots['POSITION_X'] = spots['POSITION_X'].astype(float)
        spots['POSITION_Y'] = spots['POSITION_Y'].astype(float)
        spots['TRACK_ID'] = spots['TRACK_ID'].astype(int)
 
        if 'TRACK_ID' not in tracks.columns:
            raise KeyError(f"Column 'TRACK_ID' not found in {tfile}")
        tracks['TRACK_ID'] = pd.to_numeric(tracks['TRACK_ID'], errors='coerce')
        tracks = tracks.dropna(subset=['TRACK_ID'])
        tracks['TRACK_ID'] = tracks['TRACK_ID'].astype(int)
 
        # First frame spots
        first_frame = spots['FRAME'].min()
        first_spots = spots[spots['FRAME'] == first_frame]
 
        # Determine which tracks start inside ROI
        inside_ids = []
        for _, row in first_spots.iterrows():
            x, y, tid = row['POSITION_X'], row['POSITION_Y'], row['TRACK_ID']
            col_idx = int(round(x))
            row_idx = int(round(y))
            if (0 <= row_idx < mask.shape[0] and
                0 <= col_idx < mask.shape[1] and
                mask[row_idx, col_idx]):
                inside_ids.append(tid)
 
        inside_set = set(inside_ids)
        print(f"Found {len(inside_set)} tracks beginning inside ROI.")
 
        # Filter and save
        filtered = tracks[tracks['TRACK_ID'].isin(inside_set)]
        base, ext = os.path.splitext(tfile)
        out_path = f"{base}{suffix}{ext}"
        filtered.to_csv(out_path, index=False)
        print(f"Saved filtered tracks: {os.path.basename(out_path)}")
 
if __name__ == '__main__':
    folder = input("Enter path to folder containing ROI TIFF and CSV files: ").strip()
    if not folder:
        print("Folder path is required. Exiting.")
        exit(1)
    roi = input("Enter ROI TIFF filename (or press Enter to auto-detect): ").strip() or None
    suffix = input("Enter output filename suffix (default '_filtered_tracks'): ").strip() or "_filtered_tracks"
    try:
        filter_tracks_by_roi(folder, roi_file=roi, suffix=suffix)
    except Exception as e:
        print(f"Error: {e}")